# WiDS Wildfire Survival – Anchor + Near-Fire Specialist


In [1]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

PROB_COLS = ['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']
HORIZONS = [12, 24, 48]
REQ_FILES = ['train.csv', 'test.csv', 'sample_submission.csv']


def locate_data_dir() -> Path:
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path('.'), Path('..'), Path('/mnt/data')]
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        if root.is_dir() and all((root / f).exists() for f in REQ_FILES):
            return root
        try:
            for p in root.rglob('train.csv'):
                d = p.parent
                if all((d / f).exists() for f in REQ_FILES):
                    candidates.append(d.resolve())
        except Exception:
            pass
    if not candidates:
        raise FileNotFoundError('Could not locate train.csv / test.csv / sample_submission.csv')
    candidates = sorted(set(candidates), key=lambda p: (0 if str(p).startswith('/kaggle/input') else 1, len(str(p))))
    return candidates[0]


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dist = df['dist_min_ci_0_5h'].clip(lower=1)
    df['dist_km'] = dist / 1000.0
    df['log_distance'] = np.log1p(dist)
    df['effective_speed'] = df['closing_speed_m_per_h'].clip(lower=0) + df['radial_growth_rate_m_per_h'].clip(lower=0)
    df['eta_eff'] = np.where(df['effective_speed'] > 1e-3, dist / df['effective_speed'], 9999.0)
    df['eta_eff'] = np.clip(df['eta_eff'], 0.0, 9999.0)
    df['log_eta_eff'] = np.log1p(df['eta_eff'])
    df['fire_radius_km'] = np.sqrt(df['area_first_ha'].clip(lower=0) * 10000.0 / np.pi) / 1000.0
    df['threat_eff'] = df['effective_speed'] / (df['dist_km'] + 0.2)
    df['hour_sin'] = np.sin(2.0 * np.pi * df['event_start_hour'] / 24.0)
    df['hour_cos'] = np.cos(2.0 * np.pi * df['event_start_hour'] / 24.0)
    return df


def horizon_target(df: pd.DataFrame, horizon: int):
    t = df['time_to_hit_hours'].to_numpy()
    e = df['event'].to_numpy().astype(int)
    unknown = (e == 0) & (t < horizon)
    y = ((e == 1) & (t <= horizon)).astype(int)
    usable = ~unknown
    return y, usable


def enforce_monotonicity(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, dtype=float).copy()
    arr = np.clip(arr, 0.0, 1.0)
    for j in range(1, arr.shape[1]):
        arr[:, j] = np.maximum(arr[:, j], arr[:, j - 1])
    return arr


def validate_submission(sub: pd.DataFrame, expected_ids: pd.Series) -> None:
    expected_cols = ['event_id', *PROB_COLS]
    if list(sub.columns) != expected_cols:
        raise ValueError(f'Bad columns: {list(sub.columns)}')
    if len(sub) != len(expected_ids):
        raise ValueError('Row count mismatch')
    if not sub['event_id'].is_unique:
        raise ValueError('Duplicate event_id detected')
    if sub['event_id'].tolist() != expected_ids.tolist():
        raise ValueError('event_id order/content mismatch')
    probs = sub[PROB_COLS].to_numpy(dtype=float)
    if not np.isfinite(probs).all():
        raise ValueError('Non-finite probabilities detected')
    if ((probs < 0.0) | (probs > 1.0)).any():
        raise ValueError('Probabilities out of [0,1] range')
    if (np.diff(probs, axis=1) < -1e-12).any():
        raise ValueError('Monotonicity violated')


def fit_anchor_predictions(train_df: pd.DataFrame, test_df: pd.DataFrame) -> np.ndarray:
    train_score = -train_df['dist_min_ci_0_5h'].clip(lower=1).to_numpy(dtype=float)
    test_score = -test_df['dist_min_ci_0_5h'].clip(lower=1).to_numpy(dtype=float)
    preds = np.zeros((len(test_df), 4), dtype=float)
    for j, h in enumerate(HORIZONS):
        y, usable = horizon_target(train_df, h)
        iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
        iso.fit(train_score[usable], y[usable])
        preds[:, j] = iso.predict(test_score)
    preds[:, 3] = 1.0
    return enforce_monotonicity(preds)


def fit_near_specialist(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    threshold_m: float,
    rf_depth: int,
    rf_leaf: int,
    et_depth: int,
    et_leaf: int,
    rf_weight: float,
) -> tuple[np.ndarray, np.ndarray]:
    base_features = [
        'dist_min_ci_0_5h', 'dist_km', 'log_distance', 'num_perimeters_0_5h', 'dt_first_last_0_5h',
        'low_temporal_resolution_0_5h', 'alignment_abs', 'area_growth_rate_ha_per_h',
        'radial_growth_rate_m_per_h', 'effective_speed', 'eta_eff', 'log_eta_eff', 'log1p_growth',
        'dist_std_ci_0_5h', 'dist_fit_r2_0_5h', 'fire_radius_km', 'threat_eff', 'event_start_hour',
        'hour_sin', 'hour_cos', 'spread_bearing_cos', 'spread_bearing_sin'
    ]
    features = [c for c in base_features if c in train_df.columns and c in test_df.columns]

    train_mask = train_df['dist_min_ci_0_5h'].to_numpy(dtype=float) < float(threshold_m)
    test_mask = test_df['dist_min_ci_0_5h'].to_numpy(dtype=float) < float(threshold_m)

    preds = np.full((len(test_df), 4), np.nan, dtype=float)
    preds[:, 3] = 1.0

    if train_mask.sum() == 0 or test_mask.sum() == 0:
        return preds, test_mask

    sub = train_df.loc[train_mask].reset_index(drop=True)
    X_sub = sub[features]
    X_test = test_df.loc[test_mask, features]

    for j, h in enumerate(HORIZONS):
        y, usable = horizon_target(sub, h)
        X_fit = X_sub.loc[usable]
        y_fit = y[usable]
        if len(y_fit) == 0:
            p = np.zeros(len(X_test), dtype=float)
        elif len(np.unique(y_fit)) < 2:
            p = np.full(len(X_test), float(y_fit.mean()), dtype=float)
        else:
            rf = RandomForestClassifier(
                n_estimators=280,
                max_depth=rf_depth,
                min_samples_leaf=rf_leaf,
                n_jobs=1,
                random_state=1200 + h,
            )
            et = ExtraTreesClassifier(
                n_estimators=420,
                max_depth=et_depth,
                min_samples_leaf=et_leaf,
                n_jobs=1,
                random_state=2200 + h,
            )
            rf.fit(X_fit, y_fit)
            et.fit(X_fit, y_fit)
            p = rf_weight * rf.predict_proba(X_test)[:, 1] + (1.0 - rf_weight) * et.predict_proba(X_test)[:, 1]
        preds[test_mask, j] = p

    return preds, test_mask


def build_variant(base_pred: np.ndarray, specialist_pred: np.ndarray, specialist_mask: np.ndarray, near_weight: float) -> np.ndarray:
    pred = base_pred.copy()
    if specialist_mask.any():
        pred[specialist_mask, :3] = (1.0 - near_weight) * base_pred[specialist_mask, :3] + near_weight * specialist_pred[specialist_mask, :3]
    pred[:, 3] = 1.0
    return enforce_monotonicity(pred)


def make_submission(sample_sub: pd.DataFrame, preds: np.ndarray, out_path: Path) -> pd.DataFrame:
    sub = sample_sub[['event_id']].copy()
    sub[PROB_COLS] = preds
    validate_submission(sub, sample_sub['event_id'])
    sub.to_csv(out_path, index=False)
    return sub


def main() -> None:
    data_dir = locate_data_dir()
    train = pd.read_csv(data_dir / 'train.csv')
    test = pd.read_csv(data_dir / 'test.csv')
    sample_sub = pd.read_csv(data_dir / 'sample_submission.csv')

    train = add_features(train)
    test = add_features(test)

    anchor_pred = fit_anchor_predictions(train, test)

    spec_main, mask_main = fit_near_specialist(
        train_df=train,
        test_df=test,
        threshold_m=5000,
        rf_depth=4,
        rf_leaf=1,
        et_depth=5,
        et_leaf=2,
        rf_weight=0.45,
    )
    spec_alt, mask_alt = fit_near_specialist(
        train_df=train,
        test_df=test,
        threshold_m=5000,
        rf_depth=4,
        rf_leaf=1,
        et_depth=6,
        et_leaf=2,
        rf_weight=0.50,
    )

    pred_main = build_variant(anchor_pred, spec_main, mask_main, near_weight=0.90)
    pred_hard = build_variant(anchor_pred, spec_main, mask_main, near_weight=1.00)
    pred_alt = build_variant(anchor_pred, spec_alt, mask_alt, near_weight=0.90)
    pred_anchor = anchor_pred.copy()

    make_submission(sample_sub, pred_main, Path('submission.csv'))
    make_submission(sample_sub, pred_hard, Path('submission_alt_hardreplace.csv'))
    make_submission(sample_sub, pred_alt, Path('submission_alt_etheavier.csv'))
    make_submission(sample_sub, pred_anchor, Path('submission_anchor_only.csv'))

    print('data_dir =', data_dir)
    print('test_rows =', len(test))
    print('near_5000_test_rows =', int(mask_main.sum()))
    print('wrote: submission.csv')
    print('wrote: submission_alt_hardreplace.csv')
    print('wrote: submission_alt_etheavier.csv')
    print('wrote: submission_anchor_only.csv')
    print(pd.read_csv('submission.csv').head())


if __name__ == '__main__':
    main()


data_dir = /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
test_rows = 95
near_5000_test_rows = 28
wrote: submission.csv
wrote: submission_alt_hardreplace.csv
wrote: submission_alt_etheavier.csv
wrote: submission_anchor_only.csv
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.000000  0.000000  0.000000       1.0
1  13353600  0.546336  0.949226  0.978365       1.0
2  13942327  0.000000  0.000000  0.000000       1.0
3  16112781  0.677698  0.939479  0.981446       1.0
4  17132808  0.000000  0.000000  0.000000       1.0
